<a href="https://colab.research.google.com/github/hoanganh1105/ai-ambulance-coordinator/blob/main/notebooks/Main_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cài đặt

Clone thư mục từ github, cài đặt thư viện.

In [ ]:
!git clone https://github.com/hoanganh1105/ai-ambulance-coordinator.git
%cd ai-ambulance-coordinator
!pip install -r requirements.txt

Tải dataset để train các mô hình:
- [Disease-Symptom Dataset](https://www.kaggle.com/datasets/dhivyeshrk/diseases-and-symptoms-dataset)
- [Delhi Traffic Patterns Dataset](https://www.kaggle.com/datasets/guriya79/understanding-delhi-traffic-patterns)

In [ ]:
import kagglehub
disease_classifier_train_dataset = kagglehub.dataset_download("dhivyeshrk/diseases-and-symptoms-dataset", path="Final_Augmented_dataset_Diseases_and_Symptoms.csv")
print("Path to disease dataset:", disease_classifier_train_dataset)
traffic_estimator_train_dataset = kagglehub.dataset_download("guriya79/understanding-delhi-traffic-patterns", path="delhi_traffic_features.csv")
print("Path to traffic dataset:", traffic_estimator_train_dataset)

# Khởi tạo mô hình

In [ ]:
from tabulate import tabulate
from modules.core.disease_classifier import *
from modules.core.map_router import *
from modules.core.patient_prioritizer import *
from modules.core.traffic_estimator import *

route_planner = MapRouter()
traffic_estimator = TrafficEstimator()
disease_classifier = DiseaseClassifier()
patient_prioritizer = PatientPrioritizer()

print("=== Training traffic_estimator from dataset ===")
traffic_estimator.train(traffic_estimator_train_dataset)
print("\n=== Training disease_classifier from dataset ===")
disease_classifier.train(disease_classifier_train_dataset)
print("\n=== Loading knowledgebase for patient_prioritizer ===")
patient_prioritizer.load_knowledge_base("features/knowledgebase.txt")

print("\nDone ✅.")

# Chạy thử từng module

Dự báo tình trạng giao thông

In [ ]:
# Cấu hình môi trường
time_of_day = "Night"           # "Morning Peak" / "Afternoon" / "Evening Peak" / "Night"
day_of_week = "Weekday"         # "Weekday" / "Weekend"
weather_condition = "Rain"      # "Clear" / "Rain" / "Fog" / "Heatwave"

# Hàm cập nhật trọng số.
def weight_func(data, alpha = 0.5):
  road_type = data.get("highway")
  jam_level = traffic_estimator.estimate_traffic_density_level(time_of_day,
                                                               day_of_week,
                                                               weather_condition,
                                                               road_type)
  length = data.get("length")
  data["density_level"] = jam_level
  return length * (1 + alpha * jam_level)

route_planner.add_edges_attribute("weight", weight_func)
route_planner.show_map(show_density=True)

Chẩn đoán bệnh

In [ ]:
patient_1 = Patient(['frontal headache', 'dizziness'], (28.580300971796593, 77.09896087646486))
patient_2 = Patient(['sore throat', 'hoarse voice', 'difficulty breathing'], (28.63033607110889, 77.14393615722658))
patient_3 = Patient(['bleeding from eye', 'bleeding from ear', 'fainting'],  (28.48287529214133, 77.14221954345705))
patient_4 = Patient(["fainting", "feeling ill", "vomiting blood"], (28.60630109829146, 77.19895362854005))

patient_list = [patient_1, patient_2, patient_3, patient_4]

for p in patient_list:
  p.predicted_disease = disease_classifier.predict(p.symptoms)

headers = ["ID", "Vị trí", "Triệu chứng", "Chẩn đoán"]
rows = [[p.id, p.position, ", ".join(p.symptoms), p.predicted_disease] for p in patient_list]
print(tabulate(rows, headers=headers, tablefmt="grid"))

Ưu tiên bệnh nhân

In [ ]:
most_prior_patients = patient_prioritizer.get_most_prioritized_patients(patient_list)
headers = ["ID", "Vị trí", "Triệu chứng", "Chẩn đoán"]
rows = [[p.id, p.position, ", ".join(p.symptoms), p.predicted_disease] for p in most_prior_patients]
print(tabulate(rows, headers=headers, tablefmt="grid"))

Tìm đường tới bệnh nhân được ưu tiên

In [ ]:
# Vị trí xe cứu thương
ambulance_pos = (28.60976729586756, 77.19122886657716)
# Giả sử chọn bệnh nhân đầu tiên trong danh sách ưu tiên cao nhất
chosen_patient = most_prior_patients[0]

path, _ = route_planner.optimal_path(ambulance_pos, chosen_patient.position)
route_planner.show_map(org=ambulance_pos, dests=[chosen_patient.position], route=path)

# Hệ thống tích hợp

### Cập nhật môi trường

Cấu hình hiện tại là cấu hình đã load từ các cell trước. Bỏ comment và tuỳ chỉnh nếu muốn chạy với cấu hình môi trường khác.

In [ ]:
# # Cấu hình môi trường
# time_of_day = "Night"           # "Morning Peak" / "Afternoon" / "Evening Peak" / "Night"
# day_of_week = "Weekday"         # "Weekday" / "Weekend"
# weather_condition = "Rain"      # "Clear" / "Rain" / "Fog" / "Heatwave"

# # Dự báo tình trạng giao thông
# route_planner.add_edges_attribute("weight", weight_func)

route_planner.show_map(show_density=True)

### Xử lý chính
Xử lý các yêu cầu xe cứu thương từ các bệnh nhân: hệ thống sẽ ưu tiên những bệnh nhân có triệu chứng nghiêm trọng nhất, sau đó là các bệnh nhân có bệnh được dự đoán nghiêm trọng nhất và cuối cùng là ưu tiên bệnh nhân gần nhất.

In [ ]:
def process_patients_requests(ambulance_coord: tuple[float, float], patients: list[Patient]):
  if not patients:
    print("Không có bệnh nhân để xử lý.")
    return []

  # Dự đoán bệnh của bệnh nhân
  for p in patients:
    p.predicted_disease = disease_classifier.predict(p.symptoms)

  # Danh sách các bệnh nhân được ưu tiên nhất dựa trên triệu chứng và bệnh
  most_prior_patients = patient_prioritizer.get_most_prioritized_patients(patients)

  # Tìm đường đi tới các bệnh nhân đó
  routes = [route_planner.optimal_path(ambulance_coord, p.position) for p in most_prior_patients]

  # Ưu tiên bệnh nhân gần nhất
  nearest = min(routes, key=lambda x: x[1])
  chosen_patient = most_prior_patients[routes.index(nearest)]
  path = nearest[0]

  prior_routes_info = {most_prior_patients[i]: routes[i][1] for i in range(len(most_prior_patients))}

  # Show kết quả
  print("\n=== Kết quả ===")
  print(f"\nVị trí xe cứu thương: {ambulance_coord}")
  print("Kết quả điều phối:")
  headers = ["ID", "Vị trí", "Triệu chứng", "Chẩn đoán", "Được ưu tiên", "Đường đi", "Được chọn"]
  rows = []
  for p in patients:
      is_prior = '*' if p in most_prior_patients else ''
      # Nếu nằm trong danh sách ưu tiên thì lấy chiều dài từ dictionary, format 2 chữ số thập phân
      route_len = f"{prior_routes_info[p]:.2f} m" if p in most_prior_patients else ""
      is_chosen = '*' if p is chosen_patient else ''

      rows.append([
          p.id,
          f"{p.position[0]:.4f}, {p.position[1]:.4f}", # Rút gọn hiển thị toạ độ cho bảng đẹp hơn
          ", ".join(p.symptoms),
          p.predicted_disease,
          is_prior,
          route_len,
          is_chosen
      ])
  print(tabulate(rows, headers=headers, tablefmt="grid"))
  print(f"\nĐường đi theo toạ độ: {path}")
  print("Đường đi trên bản đồ:")
  route_planner.show_map(route=path, org=ambulance_coord, dests=[p.position for p in patients])

  return path

### Giao diện tương tác

Thực hiện theo các bước sau:

1. Chọn vị trí xe cứu thương
2. Thêm một số lượng bệnh nhân tuỳ ý bằng cách lặp lại cách bước:
  - Chọn vị trí bệnh nhân
  - Liệt kê các triệu chứng
  - Nhấn "Thêm bệnh nhân"
3. Nhấn "Kết thúc và trả kết quả"

**Lưu ý**: Kết quả xuất ra màn hình có bao gồm hình ảnh bản đồ và việc render bản đồ có thể mất thời gian. Do đó, vui lòng kiên nhẫn chờ đợi hình ảnh cuối cùng.

In [ ]:
from modules.ui.dispatch_ui import *
create_dispatch_interface(route_planner.place, list(disease_classifier.symptoms_vocab), process_patients_requests)

In [ ]:
# %cd ..
# !rm -rfv ai-ambulance-coordinator